# 06 — Métricas das máximas anuais: DL × observado × coeficientes

Este notebook avalia as máximas anuais de 2024 e 2025 produzidas pelas nove configurações GRU e pelos métodos de **Basso Subquente-D** e **Dick-Peschke**, sempre usando a série observada como referência.

O arquivo de entrada já fornecido é `06_dados_maximos_anuais.txt`. Ele usa `;` como separador e mantém vazias as durações incompatíveis com cada resolução.

O notebook calcula:

- erro médio (ME) e viés percentual para identificar subestimação ou superestimação;
- MAE, RMSE, NRMSE e MAPE para medir a magnitude dos erros;
- frequência de subestimações e superestimações;
- ranking justo nas durações comuns;
- ranking separado para as saídas de 1 h, 2 h e 4 h;
- gráficos e texto automático de interpretação.

**Importante:** 24 h é o total diário usado como base pelos coeficientes e, em muitos modelos, é conservado pela desagregação. Por isso, o ranking principal usa somente durações subdiárias. O resultado com 24 h também é exportado, mas deve ser tratado como complementar.

In [ ]:
# Se faltar alguma biblioteca, descomente e execute uma vez:
# %pip install pandas numpy matplotlib seaborn

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = str

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 150)

## 1. Configuração

In [ ]:
# Deixe o TXT na mesma pasta do notebook ou informe outro caminho.
# Também são aceitos CSV e TXT com as mesmas colunas.
ARQUIVO_ENTRADA = Path("06_dados_maximos_anuais.txt")
SEPARADOR = ";"

PASTA_SAIDA = Path("resultados_metricas_06")
ANOS = [2024, 2025]
DURACOES_HORAS = [1, 2, 4, 6, 12, 24]

# Comparação possível entre todas as 9 GRUs e os 2 métodos de coeficientes.
DURACOES_COMUNS = [4, 12, 24]
DURACOES_COMUNS_SUBDIARIAS = [4, 12]

# Comparação de cada resolução DL com Basso e Dick-Peschke.
# Exclui 24 h para avaliar efetivamente a desagregação temporal.
DURACOES_SUBDIARIAS_POR_RESOLUCAO = {
    1: [1, 2, 4, 6, 12],
    2: [2, 4, 6, 12],
    4: [4, 12],
}

TOLERANCIA_IGUAL_MM = 1e-9
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

## 2. Leitura e validação dos dados

In [ ]:
def converter_numero(serie):
    texto = serie.astype("string").str.strip()
    ambos = texto.str.contains(",", na=False) & texto.str.contains(r"\.", na=False)
    texto.loc[ambos] = texto.loc[ambos].str.replace(".", "", regex=False)
    texto = texto.str.replace(",", ".", regex=False)
    return pd.to_numeric(texto, errors="coerce")


if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ARQUIVO_ENTRADA.resolve()}\n"
        "Coloque 06_dados_maximos_anuais.txt na mesma pasta do notebook "
        "ou altere ARQUIVO_ENTRADA."
    )

dados = pd.read_csv(
    ARQUIVO_ENTRADA,
    sep=SEPARADOR,
    dtype=str,
    encoding="utf-8-sig",
    keep_default_na=False,
)
dados.columns = dados.columns.str.strip()

colunas_maximos = [f"max_{d}h_mm" for d in DURACOES_HORAS]
obrigatorias = ["serie", "tipo", "resolucao_h", "janela_dias", "ano", *colunas_maximos]
faltantes = [c for c in obrigatorias if c not in dados.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes no arquivo: {faltantes}")

dados["serie"] = dados["serie"].astype("string").str.strip()
dados["tipo"] = dados["tipo"].astype("string").str.strip().str.lower()
dados["ano"] = converter_numero(dados["ano"]).astype("Int64")
dados["resolucao_h"] = converter_numero(dados["resolucao_h"]).astype("Int64")
dados["janela_dias"] = converter_numero(dados["janela_dias"]).astype("Int64")
for coluna in colunas_maximos:
    dados[coluna] = converter_numero(dados[coluna])

# Completa o tipo quando o usuário deixar a célula vazia.
dados.loc[dados["serie"].str.upper().eq("OBSERVADA"), "tipo"] = "observada"
dados.loc[dados["serie"].str.upper().str.startswith("GRU_"), "tipo"] = "simulada"
dados.loc[~dados["tipo"].isin(["observada", "simulada"]), "tipo"] = "coeficiente"

if dados[["serie", "ano"]].duplicated().any():
    repetidas = dados.loc[dados[["serie", "ano"]].duplicated(False), ["serie", "ano"]]
    raise ValueError("Existem pares série/ano duplicados:\n" + repetidas.to_string(index=False))

anos_lidos = set(dados["ano"].dropna().astype(int))
if not set(ANOS).issubset(anos_lidos):
    raise ValueError(f"Faltam anos. Esperados: {ANOS}; encontrados: {sorted(anos_lidos)}")

observada_larga = dados.query("tipo == 'observada'")
if len(observada_larga) != len(ANOS) or set(observada_larga["ano"].astype(int)) != set(ANOS):
    raise ValueError("Deve existir exatamente uma linha observada para cada ano de análise.")
if observada_larga[colunas_maximos].isna().any().any():
    raise ValueError("A série observada precisa conter todas as durações.")
if (dados[colunas_maximos] < 0).any().any():
    raise ValueError("Foram encontrados valores negativos de precipitação máxima.")

print(f"Arquivo: {ARQUIVO_ENTRADA.resolve()}")
print(f"Linhas: {len(dados)} | Séries: {dados['serie'].nunique()} | Anos: {sorted(anos_lidos)}")
display(dados)

## 3. Tabela longa e erros caso a caso

In [ ]:
dados_longos = dados.melt(
    id_vars=["serie", "tipo", "resolucao_h", "janela_dias", "ano"],
    value_vars=colunas_maximos,
    var_name="duracao_coluna",
    value_name="estimado_mm",
)
dados_longos["duracao_h"] = (
    dados_longos["duracao_coluna"].str.extract(r"max_(\d+)h_mm")[0].astype(int)
)
dados_longos = dados_longos.drop(columns="duracao_coluna")

observado = dados_longos.query("tipo == 'observada'")[[
    "ano", "duracao_h", "estimado_mm"
]].rename(columns={"estimado_mm": "observado_mm"})

detalhes = dados_longos.query("tipo != 'observada'").dropna(subset=["estimado_mm"]).merge(
    observado,
    on=["ano", "duracao_h"],
    how="left",
    validate="many_to_one",
)
if detalhes["observado_mm"].isna().any():
    raise ValueError("Existem estimativas sem valor observado correspondente.")

detalhes["grupo"] = np.where(detalhes["tipo"].eq("simulada"), "DL", "Coeficiente")
detalhes["erro_mm"] = detalhes["estimado_mm"] - detalhes["observado_mm"]
detalhes["erro_absoluto_mm"] = detalhes["erro_mm"].abs()
detalhes["erro_pct"] = np.where(
    detalhes["observado_mm"].abs() > 1e-12,
    100 * detalhes["erro_mm"] / detalhes["observado_mm"],
    np.nan,
)
detalhes["erro_absoluto_pct"] = detalhes["erro_pct"].abs()
detalhes["razao_estimado_observado"] = np.where(
    detalhes["observado_mm"].abs() > 1e-12,
    detalhes["estimado_mm"] / detalhes["observado_mm"],
    np.nan,
)
detalhes["situacao"] = np.select(
    [
        detalhes["erro_mm"] < -TOLERANCIA_IGUAL_MM,
        detalhes["erro_mm"] > TOLERANCIA_IGUAL_MM,
    ],
    ["subestimou", "superestimou"],
    default="igual",
)
detalhes = detalhes.sort_values(["grupo", "serie", "ano", "duracao_h"]).reset_index(drop=True)
display(detalhes.head(40))

## 4. Métricas utilizadas

Para cada estimativa (S_i) e observação (O_i), o erro é definido como:

\[
e_i=S_i-O_i
\]

Portanto, **erro negativo indica subestimação** e **erro positivo indica superestimação**.

| Métrica | Interpretação |
|---|---|
| ME ou erro médio (mm) | Direção do viés; negativo subestima e positivo superestima |
| MAE (mm) | Magnitude média do erro, sem cancelamento de sinais |
| RMSE (mm) | Penaliza mais os erros grandes; principal medida de precisão |
| NRMSE (%) | RMSE dividido pela média observada; facilita comparação relativa |
| MAPE (%) | Erro percentual absoluto médio |
| Viés (%) | Soma dos erros dividida pela soma observada; positivo superestima |
| Frequência (%) | Percentual de casos subestimados, superestimados ou iguais |

Com apenas dois anos, correlação, (R^2), NSE e KGE não são usados como evidência principal. Além da amostra pequena, juntar durações diferentes pode criar correlação artificial apenas porque acumulados maiores ocorrem em durações maiores.

In [ ]:
def calcular_metricas(bloco):
    bloco = bloco.dropna(subset=["estimado_mm", "observado_mm"]).copy()
    if bloco.empty:
        return {}
    erro = bloco["estimado_mm"] - bloco["observado_mm"]
    obs = bloco["observado_mm"]
    validos_pct = obs.abs() > 1e-12
    erro_pct = 100 * erro.loc[validos_pct] / obs.loc[validos_pct]
    rmse = float(np.sqrt(np.mean(np.square(erro))))
    media_obs = float(obs.mean())
    soma_obs = float(obs.sum())
    return {
        "n": int(len(bloco)),
        "erro_medio_mm": float(erro.mean()),
        "mae_mm": float(erro.abs().mean()),
        "rmse_mm": rmse,
        "nrmse_pct": 100 * rmse / media_obs if abs(media_obs) > 1e-12 else np.nan,
        "mape_pct": float(erro_pct.abs().mean()) if len(erro_pct) else np.nan,
        "vies_pct": 100 * float(erro.sum()) / soma_obs if abs(soma_obs) > 1e-12 else np.nan,
        "erro_pct_medio": float(erro_pct.mean()) if len(erro_pct) else np.nan,
        "erro_pct_mediano": float(erro_pct.median()) if len(erro_pct) else np.nan,
        "razao_media": float(bloco["razao_estimado_observado"].mean()),
        "subestimacoes_n": int((erro < -TOLERANCIA_IGUAL_MM).sum()),
        "superestimacoes_n": int((erro > TOLERANCIA_IGUAL_MM).sum()),
        "iguais_n": int((erro.abs() <= TOLERANCIA_IGUAL_MM).sum()),
        "subestimacoes_pct": 100 * float((erro < -TOLERANCIA_IGUAL_MM).mean()),
        "superestimacoes_pct": 100 * float((erro > TOLERANCIA_IGUAL_MM).mean()),
        "iguais_pct": 100 * float((erro.abs() <= TOLERANCIA_IGUAL_MM).mean()),
    }


def resumir_por(tabela, chaves):
    linhas = []
    agrupador = chaves[0] if len(chaves) == 1 else chaves
    for valores, bloco in tabela.groupby(agrupador, dropna=False, sort=False):
        if len(chaves) == 1:
            valores = (valores,)
        elif not isinstance(valores, tuple):
            valores = (valores,)
        linha = dict(zip(chaves, valores))
        linha.update(calcular_metricas(bloco))
        linhas.append(linha)
    return pd.DataFrame(linhas)


CHAVES_MODELO = ["serie", "grupo", "resolucao_h", "janela_dias"]
metricas_todas = resumir_por(detalhes, CHAVES_MODELO).sort_values("rmse_mm")
metricas_subdiarias = resumir_por(
    detalhes.query("duracao_h < 24"), CHAVES_MODELO
).sort_values("rmse_mm")

print("Métricas usando todas as durações disponíveis de cada método:")
display(metricas_todas.round(3))
print("Métricas somente subdiárias disponíveis de cada método:")
display(metricas_subdiarias.round(3))

## 5. Ranking comum entre todos os métodos

In [ ]:
def criar_ranking_comum(duracoes, nome_escopo):
    base = detalhes[detalhes["duracao_h"].isin(duracoes)].copy()
    resumo = resumir_por(base, CHAVES_MODELO)
    n_esperado = len(ANOS) * len(duracoes)
    incompletos = resumo.loc[resumo["n"] != n_esperado, ["serie", "n"]]
    if not incompletos.empty:
        raise ValueError(
            f"Séries incompletas no escopo {nome_escopo}:\n{incompletos.to_string(index=False)}"
        )
    resumo["escopo"] = nome_escopo
    resumo["duracoes_h"] = ",".join(map(str, duracoes))
    resumo = resumo.sort_values(["rmse_mm", "mae_mm", "mape_pct"]).reset_index(drop=True)
    resumo.insert(0, "posicao_rmse", np.arange(1, len(resumo) + 1))
    return resumo


# Ranking principal: exclui 24 h, que é a chuva diária de partida.
ranking_comum_subdiario = criar_ranking_comum(
    DURACOES_COMUNS_SUBDIARIAS,
    "comum_subdiario_4h_12h",
)

# Ranking complementar: inclui 24 h.
ranking_comum_com_24h = criar_ranking_comum(
    DURACOES_COMUNS,
    "comum_4h_12h_24h",
)

print("RANKING PRINCIPAL — durações comuns subdiárias de 4 e 12 h:")
display(ranking_comum_subdiario.round(3))
print("RANKING COMPLEMENTAR — 4, 12 e 24 h:")
display(ranking_comum_com_24h.round(3))

## 6. Ranking justo por resolução de saída

In [ ]:
rankings_resolucao = []
for resolucao, duracoes in DURACOES_SUBDIARIAS_POR_RESOLUCAO.items():
    eh_dl_resolucao = detalhes["grupo"].eq("DL") & detalhes["resolucao_h"].eq(resolucao)
    eh_coeficiente = detalhes["grupo"].eq("Coeficiente")
    base = detalhes.loc[
        (eh_dl_resolucao | eh_coeficiente) & detalhes["duracao_h"].isin(duracoes)
    ].copy()
    resumo = resumir_por(base, CHAVES_MODELO)
    n_esperado = len(ANOS) * len(duracoes)
    resumo = resumo.loc[resumo["n"] == n_esperado].copy()
    resumo["resolucao_avaliada_h"] = resolucao
    resumo["duracoes_subdiarias_h"] = ",".join(map(str, duracoes))
    resumo = resumo.sort_values(["rmse_mm", "mae_mm"]).reset_index(drop=True)
    resumo["posicao_rmse"] = np.arange(1, len(resumo) + 1)
    rankings_resolucao.append(resumo)

ranking_por_resolucao = pd.concat(rankings_resolucao, ignore_index=True)
ranking_por_resolucao = ranking_por_resolucao[[
    "resolucao_avaliada_h", "duracoes_subdiarias_h", "posicao_rmse",
    *CHAVES_MODELO, *[c for c in ranking_por_resolucao.columns if c not in {
        "resolucao_avaliada_h", "duracoes_subdiarias_h", "posicao_rmse", *CHAVES_MODELO
    }]
]]
display(ranking_por_resolucao.round(3))

melhores_dl_resolucao = (
    ranking_por_resolucao.query("grupo == 'DL'")
    .sort_values(["resolucao_avaliada_h", "rmse_mm"])
    .groupby("resolucao_avaliada_h", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

comparacoes_melhores = []
for _, melhor in melhores_dl_resolucao.iterrows():
    resolucao = int(melhor["resolucao_avaliada_h"])
    coeficientes = ranking_por_resolucao.query(
        "resolucao_avaliada_h == @resolucao and grupo == 'Coeficiente'"
    )
    for _, coef in coeficientes.iterrows():
        comparacoes_melhores.append({
            "resolucao_h": resolucao,
            "melhor_dl": melhor["serie"],
            "rmse_dl_mm": melhor["rmse_mm"],
            "metodo_coeficiente": coef["serie"],
            "rmse_coeficiente_mm": coef["rmse_mm"],
            "melhoria_rmse_dl_pct": (
                100 * (coef["rmse_mm"] - melhor["rmse_mm"]) / coef["rmse_mm"]
                if abs(coef["rmse_mm"]) > 1e-12 else np.nan
            ),
            "mae_dl_mm": melhor["mae_mm"],
            "mae_coeficiente_mm": coef["mae_mm"],
        })
comparacao_melhores_dl = pd.DataFrame(comparacoes_melhores)

print("Melhor configuração DL de cada resolução:")
display(melhores_dl_resolucao.round(3))
print("Redução do RMSE do melhor DL em relação aos coeficientes (positivo favorece DL):")
display(comparacao_melhores_dl.round(3))

## 7. Diagnóstico por duração e por ano

In [ ]:
metricas_por_duracao = resumir_por(
    detalhes,
    [*CHAVES_MODELO, "duracao_h"],
).sort_values(["duracao_h", "rmse_mm"])

metricas_por_ano = resumir_por(
    detalhes,
    [*CHAVES_MODELO, "ano"],
).sort_values(["ano", "rmse_mm"])

resumo_grupos_modelos = (
    ranking_comum_subdiario.groupby("grupo")[[
        "erro_medio_mm", "mae_mm", "rmse_mm", "nrmse_pct", "mape_pct",
        "vies_pct", "subestimacoes_pct", "superestimacoes_pct"
    ]]
    .agg(["mean", "median", "min", "max"])
    .round(3)
)

print("Métricas por método e duração — cada linha usa os dois anos:")
display(metricas_por_duracao.round(3))
print("Métricas por método e ano — apenas diagnóstico, pois o número de durações varia:")
display(metricas_por_ano.round(3))
print("Resumo dos modelos no escopo comum subdiário de 4 e 12 h:")
display(resumo_grupos_modelos)

## 8. Gráficos

In [ ]:
# 8.1 Ranking de RMSE nas durações subdiárias comuns (4 e 12 h)
ordem = ranking_comum_subdiario.sort_values("rmse_mm", ascending=True)
cores = ordem["grupo"].map({"DL": "#2C7FB8", "Coeficiente": "#D95F0E"})

fig, ax = plt.subplots(figsize=(10, 6.5))
barras = ax.barh(ordem["serie"], ordem["rmse_mm"], color=cores)
ax.invert_yaxis()
ax.set_xlabel("RMSE (mm)")
ax.set_ylabel("")
ax.set_title("Ranking nas durações subdiárias comuns: 4 e 12 h")
for barra, valor in zip(barras, ordem["rmse_mm"]):
    ax.text(valor + max(ordem["rmse_mm"]) * 0.01, barra.get_y() + barra.get_height()/2,
            f"{valor:.1f}", va="center", fontsize=9)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#2C7FB8", label="DL"), Patch(color="#D95F0E", label="Coeficiente")])
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "ranking_rmse_comum_4h_12h.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# 8.2 Erro percentual médio de 2024–2025: azul subestima, vermelho superestima
matriz_erro = detalhes.pivot_table(
    index="serie", columns="duracao_h", values="erro_pct", aggfunc="mean"
).reindex(columns=DURACOES_HORAS)
ordem_heatmap = ranking_comum_subdiario["serie"].tolist()
matriz_erro = matriz_erro.reindex(ordem_heatmap)
limite = np.nanmax(np.abs(matriz_erro.to_numpy()))

fig, ax = plt.subplots(figsize=(11, 7.5))
sns.heatmap(
    matriz_erro,
    cmap="RdBu_r",
    center=0,
    vmin=-limite,
    vmax=limite,
    annot=True,
    fmt=".1f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Erro percentual médio (%)"},
    ax=ax,
)
ax.set_xlabel("Duração (h)")
ax.set_ylabel("")
ax.set_title("Direção do erro médio em 2024–2025")
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "mapa_erros_percentuais.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# 8.3 Observado, coeficientes e melhor DL de cada resolução
series_destaque = ["OBSERVADA", "Basso Subquente-D", "Dick-Peschke"]
series_destaque += melhores_dl_resolucao["serie"].drop_duplicates().tolist()
curvas = dados_longos[dados_longos["serie"].isin(series_destaque)].dropna(subset=["estimado_mm"])

fig, eixos = plt.subplots(1, len(ANOS), figsize=(14, 5.2), sharey=True)
if len(ANOS) == 1:
    eixos = [eixos]
paleta = {
    "OBSERVADA": "#111111",
    "Basso Subquente-D": "#D95F0E",
    "Dick-Peschke": "#E7298A",
}
cores_dl = ["#1B9E77", "#377EB8", "#6A3D9A"]
for serie, cor in zip(melhores_dl_resolucao["serie"].drop_duplicates(), cores_dl):
    paleta[serie] = cor

for ax, ano in zip(eixos, ANOS):
    base_ano = curvas[curvas["ano"].eq(ano)]
    for serie in series_destaque:
        linha = base_ano[base_ano["serie"].eq(serie)].sort_values("duracao_h")
        if linha.empty:
            continue
        estilo = "--" if serie in ["Basso Subquente-D", "Dick-Peschke"] else "-"
        largura = 2.8 if serie == "OBSERVADA" else 1.9
        ax.plot(
            linha["duracao_h"], linha["estimado_mm"], marker="o",
            linewidth=largura, linestyle=estilo, color=paleta.get(serie), label=serie,
        )
    ax.set_xticks(DURACOES_HORAS)
    ax.set_xlabel("Duração (h)")
    ax.set_title(str(ano))
eixos[0].set_ylabel("Máxima anual acumulada (mm)")
handles, labels = eixos[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.suptitle("Máximas anuais: observado, coeficientes e melhores configurações DL", y=1.02)
fig.tight_layout(rect=[0, 0.13, 1, 1])
fig.savefig(PASTA_SAIDA / "curvas_comparativas.png", dpi=220, bbox_inches="tight")
plt.show()

## 9. Interpretação automática

In [ ]:
melhor_dl_comum = ranking_comum_subdiario.query("grupo == 'DL'").iloc[0]
coef_comum = ranking_comum_subdiario.query("grupo == 'Coeficiente'").set_index("serie")
coef_sub = metricas_subdiarias.query("grupo == 'Coeficiente'").set_index("serie")

linhas_resumo = [
    "RESUMO AUTOMÁTICO — MÁXIMAS ANUAIS 2024–2025",
    "",
    "1. O ranking principal exclui 24 h e usa 4 e 12 h, as durações subdiárias disponíveis em todas as resoluções.",
    (
        f"2. Melhor configuração DL no escopo comum: {melhor_dl_comum['serie']}, "
        f"RMSE = {melhor_dl_comum['rmse_mm']:.2f} mm, MAE = {melhor_dl_comum['mae_mm']:.2f} mm, "
        f"ME = {melhor_dl_comum['erro_medio_mm']:.2f} mm e viés = {melhor_dl_comum['vies_pct']:.2f}%."
    ),
]

numero_item = 3
for metodo in ["Basso Subquente-D", "Dick-Peschke"]:
    if metodo in coef_sub.index:
        m = coef_sub.loc[metodo]
        comum = coef_comum.loc[metodo]
        reducao_rmse = 100 * (comum["rmse_mm"] - melhor_dl_comum["rmse_mm"]) / comum["rmse_mm"]
        linhas_resumo.append(
            f"{numero_item}. {metodo}, considerando todas as durações subdiárias, apresentou "
            f"RMSE = {m['rmse_mm']:.2f} mm, ME = {m['erro_medio_mm']:.2f} mm, "
            f"viés = {m['vies_pct']:.2f}% e superestimou {m['superestimacoes_pct']:.1f}% dos casos. "
            f"Nas durações comuns de 4 e 12 h, o melhor DL reduziu o RMSE em "
            f"{reducao_rmse:.1f}% em relação a esse método."
        )
        numero_item += 1

linhas_resumo.append("")
linhas_resumo.append("Melhores configurações DL por resolução:")
for _, linha in melhores_dl_resolucao.iterrows():
    direcao = "subestimação" if linha["erro_medio_mm"] < 0 else "superestimação"
    linhas_resumo.append(
        f"- {int(linha['resolucao_avaliada_h'])} h: {linha['serie']} — "
        f"RMSE {linha['rmse_mm']:.2f} mm, ME {linha['erro_medio_mm']:.2f} mm "
        f"({direcao})."
    )

linhas_resumo += [
    "",
    "Leitura metodológica:",
    "- ME e viés negativos indicam subestimação; positivos indicam superestimação.",
    "- RMSE e MAE menores indicam maior proximidade das máximas observadas.",
    "- O resultado de 24 h não deve sustentar sozinho a superioridade de um método, pois é a escala diária de partida.",
    "- Como existem somente dois anos, os resultados são descritivos do teste de 2024–2025 e não demonstram significância estatística nem desempenho climatológico de longo prazo.",
]

texto_resumo = "\n".join(linhas_resumo)
print(texto_resumo)
(PASTA_SAIDA / "resumo_interpretacao.txt").write_text(texto_resumo, encoding="utf-8")

## 10. Exportação dos resultados

In [ ]:
arquivos_saida = {
    "erros_detalhados.csv": detalhes,
    "metricas_todas_duracoes_disponiveis.csv": metricas_todas,
    "metricas_subdiarias_disponiveis.csv": metricas_subdiarias,
    "ranking_comum_subdiario_4h_12h.csv": ranking_comum_subdiario,
    "ranking_comum_4h_12h_24h.csv": ranking_comum_com_24h,
    "ranking_subdiario_por_resolucao.csv": ranking_por_resolucao,
    "melhores_dl_por_resolucao.csv": melhores_dl_resolucao,
    "comparacao_melhores_dl_coeficientes.csv": comparacao_melhores_dl,
    "metricas_por_duracao.csv": metricas_por_duracao,
    "metricas_por_ano.csv": metricas_por_ano,
}

for nome, tabela in arquivos_saida.items():
    caminho = PASTA_SAIDA / nome
    tabela.to_csv(caminho, sep=";", decimal=",", index=False, encoding="utf-8-sig")

print("Arquivos criados em:", PASTA_SAIDA.resolve())
for caminho in sorted(PASTA_SAIDA.iterdir()):
    print(" -", caminho.name)

## Como escrever os resultados

Use o **ranking comum subdiário de 4 e 12 h** para afirmar qual configuração apresentou os menores erros em uma comparação totalmente equivalente. Use o **ranking por resolução** para discutir separadamente as saídas de 1 h, 2 h e 4 h.

Uma formulação adequada é:

> No período de teste de 2024–2025, as configurações de aprendizado profundo apresentaram comportamento predominantemente subestimador nas menores durações, enquanto os métodos baseados em coeficientes mostraram viés positivo e superestimação mais frequente. A comparação quantitativa foi realizada por ME, MAE, RMSE, NRMSE, MAPE e viés percentual. Como a amostra contém apenas duas máximas anuais por duração, os resultados devem ser interpretados como uma avaliação descritiva dos eventos do período de teste.

Substitua “apresentaram melhores resultados” pelos valores concretos do ranking. Não generalize para toda a climatologia de Estrela usando somente 2024 e 2025.